In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Load the data generated from the geolocation step
df = pd.read_csv('../data/processed/ecommerce_cleaned_geo.csv')
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

In [2]:
print("Engineering behavioral and temporal features...")

# 1. Time-Since-Signup (Converted to hours as a float)
# Fraud Pattern: Bots buy things immediately after automating account creation.
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 3600.0

# 2. Temporal Features
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek

# 3. Device Shared Velocity
# Count how many unique users are sharing the exact same device ID
df['user_count_per_device'] = df.groupby('device_id')['user_id'].transform('count')

# 4. IP Shared Velocity
# Count how many unique users are sharing the exact same IP Address
df['user_count_per_ip'] = df.groupby('ip_address')['user_id'].transform('count')

print("New behavioral features successfully created.")

Engineering behavioral and temporal features...
New behavioral features successfully created.


In [3]:
# Drop unique IDs and raw timestamps that models can't generalize from
columns_to_drop = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address']
df_ml = df.drop(columns=columns_to_drop)

# Handle high-cardinality categorical features: Top 15 countries, group rest as 'Other'
top_countries = df_ml['country'].value_counts().index[:15]
df_ml['country'] = df_ml['country'].apply(lambda x: x if x in top_countries else 'Other')

# Perform One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df_ml, columns=['source', 'browser', 'sex', 'country'], drop_first=True)

# Separate independent features (X) from the target label (y)
X = df_encoded.drop(columns=['class'])
y = df_encoded['class']

In [4]:
# Split data into 80% Training and 20% Testing sets
# stratify=y preserves the exact fraud-to-legitimate ratio in both subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale numerical columns based ONLY on training metrics to avoid data leakage
numerical_cols = ['purchase_value', 'age', 'time_since_signup', 'user_count_per_device', 'user_count_per_ip']

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

Train Shape: (120889, 30), Test Shape: (30223, 30)


In [6]:
import pandas as pd
import numpy as np

def ip_to_int(ip_series):
    """Converts a string IP address to an unsigned 32-bit integer."""
    # Handle both string IPs and float/int data if already modified
    if ip_series.dtype == np.float64 or ip_series.dtype == np.int64:
        return ip_series.astype(np.int64)
        
    # Split across dots, expand to columns, shift bits and sum
    components = ip_series.str.split('.', expand=True).astype(float).fillna(0).astype(np.int64)
    return (components[0] << 24) + (components[1] << 16) + (components[2] << 8) + components[3]

# 1. Load Datasets
df_fraud = pd.read_csv('../data/raw/Fraud_Data.csv')
df_ip = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

# 2. Preprocess IPs for merge
df_fraud['ip_int'] = ip_to_int(df_fraud['ip_address'])
df_ip['lower_bound_ip_address'] = df_ip['lower_bound_ip_address'].astype(np.int64)
df_ip['upper_bound_ip_address'] = df_ip['upper_bound_ip_address'].astype(np.int64)

# Sort both sets for range-based matching
df_fraud = df_fraud.sort_values('ip_int')
df_ip = df_ip.sort_values('lower_bound_ip_address')

# 3. Perform Range Match via merge_asof
df_merged = pd.merge_asof(
    df_fraud, 
    df_ip, 
    left_on='ip_int', 
    right_on='lower_bound_ip_address', 
    direction='backward'
)

# Validate if the IP sits inside the matched bound, otherwise flag Unknown
df_merged['country'] = np.where(
    df_merged['ip_int'] <= df_merged['upper_bound_ip_address'], 
    df_merged['country'], 
    'Unknown'
)

# Clean up temporary lookup columns
df_merged.drop(columns=['lower_bound_ip_address', 'upper_bound_ip_address'], inplace=True)
print(f"Class Imbalance:\n{df_merged['class'].value_counts(normalize=True) * 100}")

Class Imbalance:
class
0    90.635423
1     9.364577
Name: proportion, dtype: float64


In [8]:
# Convert timestamps
df_merged['signup_time'] = pd.to_datetime(df_merged['signup_time'])
df_merged['purchase_time'] = pd.to_datetime(df_merged['purchase_time'])

# 1. Time since signup (in hours)
df_merged['time_since_signup'] = (df_merged['purchase_time'] - df_merged['signup_time']).dt.total_seconds() / 3600.0

# 2. Temporal Extractions
df_merged['hour_of_day'] = df_merged['purchase_time'].dt.hour
df_merged['day_of_week'] = df_merged['purchase_time'].dt.dayofweek

# 3. Device Velocity (Transaction frequency within short windows)
df_merged = df_merged.sort_values('purchase_time')


# Save processed dataset
df_merged.to_csv('../data/processed/fraud_data_engineered.csv', index=False)